# Part 2 — Step 8: Domain Adaptation

ACNE04 (face selfies) and DermNet (clinical photos) differ in:
- **Color/lighting** — clinical photos have controlled, cooler lighting vs warm selfie tones
- **Scale** — clinical images are tighter crops, selfies show more context
- **Sharpness** — clinical images vary more in focus

We address the domain gap through **augmentation-based adaptation** applied during training:

| Technique | Purpose |
|---|---|
| `ColorJitter(brightness, contrast, saturation, hue)` | Robust to lighting differences |
| `RandomResizedCrop(scale=0.7-1.0)` | Robust to scale/zoom variation |
| `GaussianBlur` | Robust to sharpness differences |
| `RandomHorizontalFlip` | Invariance to left/right orientation |

**Prerequisites:** Run `06_patch_extraction.ipynb` and `07_train_classifier.ipynb` first.

**This notebook:** Visualises the domain gap and shows augmented vs original patches.

In [ ]:
import os
from pathlib import Path

if Path('/content').exists():
    os.chdir('/content/AcneDetection')
print(f'Working directory: {os.getcwd()}')

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from torchvision import transforms

%matplotlib inline

PATCH_DIR   = Path('data/patches')
DERMNET_DIR = Path('data/dermnet')
ACNE_FOLDER = 'Acne and Rosacea Photos'
OUT_FIG     = Path('outputs/figures')
OUT_FIG.mkdir(parents=True, exist_ok=True)

random.seed(42)

## 1. Visualise the domain gap

Side-by-side comparison of ACNE04 patches vs DermNet acne images.

In [ ]:
acne04_samples  = random.sample(list((PATCH_DIR / 'train' / 'acne').glob('*.jpg')), 4)
dermnet_samples = random.sample(list((DERMNET_DIR / 'train' / ACNE_FOLDER).glob('*')), 4)

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for col, f in enumerate(acne04_samples):
    axes[0][col].imshow(Image.open(f))
    axes[0][col].set_title('ACNE04 patch', fontsize=9)
    axes[0][col].axis('off')
for col, f in enumerate(dermnet_samples):
    axes[1][col].imshow(Image.open(f))
    axes[1][col].set_title('DermNet acne', fontsize=9)
    axes[1][col].axis('off')

plt.suptitle('Domain Gap — ACNE04 (top) vs DermNet (bottom)', fontsize=13)
plt.tight_layout()
plt.savefig(OUT_FIG / 'domain_gap.png', dpi=150)
plt.show()
print('Saved -> outputs/figures/domain_gap.png')

## 2. Visualise training augmentation

Shows how the same ACNE04 patch looks after augmentation — simulating the variety
of lighting/color/scale conditions the model sees during training.

In [ ]:
aug_tf = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.3, hue=0.05),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.5)),
])

src_img = Image.open(acne04_samples[0]).convert('RGB')

fig, axes = plt.subplots(1, 6, figsize=(16, 3))
axes[0].imshow(src_img)
axes[0].set_title('Original', fontsize=9)
axes[0].axis('off')

for i in range(1, 6):
    axes[i].imshow(aug_tf(src_img))
    axes[i].set_title(f'Augmented {i}', fontsize=9)
    axes[i].axis('off')

plt.suptitle('Training Augmentation — same patch, different random transforms', fontsize=12)
plt.tight_layout()
plt.savefig(OUT_FIG / 'augmentation_examples.png', dpi=150)
plt.show()
print('Saved -> outputs/figures/augmentation_examples.png')

## 3. Channel statistics comparison

Quantifies the domain gap by comparing mean RGB values between ACNE04 and DermNet.
Large differences confirm why domain adaptation is needed.

In [ ]:
def channel_stats(paths, n=200):
    """Compute mean R, G, B across a sample of images."""
    sample = random.sample(list(paths), min(n, len(list(paths))))
    means = []
    for p in sample:
        arr = np.array(Image.open(p).convert('RGB').resize((224, 224)), dtype=float)
        means.append(arr.mean(axis=(0, 1)) / 255.0)
    means = np.array(means)
    return means.mean(axis=0), means.std(axis=0)

acne04_paths  = list((PATCH_DIR / 'train' / 'acne').glob('*.jpg'))
dermnet_paths = list((DERMNET_DIR / 'train' / ACNE_FOLDER).glob('*'))

a_mean, a_std = channel_stats(acne04_paths)
d_mean, d_std = channel_stats(dermnet_paths)

channels = ['R', 'G', 'B']
x = np.arange(3)
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(x - 0.2, a_mean, 0.35, yerr=a_std, label='ACNE04', color='#FF6B6B', capsize=4)
ax.bar(x + 0.2, d_mean, 0.35, yerr=d_std, label='DermNet', color='#4ECDC4', capsize=4)
ax.set_xticks(x); ax.set_xticklabels(channels)
ax.set_ylabel('Mean pixel value (0-1)')
ax.set_title('RGB Channel Statistics — ACNE04 vs DermNet')
ax.legend()
plt.tight_layout()
plt.savefig(OUT_FIG / 'channel_stats.png', dpi=150)
plt.show()

print('ACNE04  mean RGB:', a_mean.round(3))
print('DermNet mean RGB:', d_mean.round(3))
print('Difference      :', (d_mean - a_mean).round(3))